In [ ]:

import sys
import pandas as pd
import numpy as np
import matplotlib
import sklearn

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\nASTRA HEALTH - Veri inceleme ortamı hazır!")


In [ ]:

from pathlib import Path

# Proje ana klasörünü bul
PROJECT_ROOT = Path.cwd().parent

# NASA veri dosyalarının yolları
CBC_FILE = PROJECT_ROOT / "data" / "raw" / "OSD-569" / "LSDS-7_Complete_Blood_Count_CBC.upload_SUBMITTED.csv"

CMP_FILE = PROJECT_ROOT / "data" / "raw" / "OSD-575" / "LSDS-8_Comprehensive_Metabolic_Panel_CMP.upload_SUBMITTED.csv"

# Dosyaların varlığını kontrol et
print("CBC dosyası mevcut mu:", CBC_FILE.exists())
print("CMP dosyası mevcut mu:", CMP_FILE.exists())

print("\nCBC yolu:", CBC_FILE)
print("CMP yolu:", CMP_FILE)



In [ ]:

# NASA CSV dosyalarını oku
cbc_df = pd.read_csv(CBC_FILE)
cmp_df = pd.read_csv(CMP_FILE)

# 1. CBC veri setini incele
print("=" * 60)
print("OSD-569 - COMPLETE BLOOD COUNT (CBC)")
print("=" * 60)

print("\nVeri boyutu (satır, sütun):", cbc_df.shape)

print("\nSütun isimleri:")
print(cbc_df.columns.tolist())

print("\nİlk 5 kayıt:")
display(cbc_df.head())

print("\nVeri türleri:")
print(cbc_df.dtypes)

print("\nEksik değer sayıları:")
print(cbc_df.isnull().sum())


# 2. CMP veri setini incele
print("\n" + "=" * 60)
print("OSD-575 - COMPREHENSIVE METABOLIC PANEL (CMP)")
print("=" * 60)

print("\nVeri boyutu (satır, sütun):", cmp_df.shape)

print("\nSütun isimleri:")
print(cmp_df.columns.tolist())

print("\nİlk 5 kayıt:")
display(cmp_df.head())

print("\nVeri türleri:")
print(cmp_df.dtypes)

print("\nEksik değer sayıları:")
print(cmp_df.isnull().sum())


In [ ]:

# 1. Katılımcı sayıları
print("CBC katılımcı sayısı:", cbc_df["SUBJECT_ID"].nunique())
print("CMP katılımcı sayısı:", cmp_df["SUBJECT_ID"].nunique())

# 2. Katılımcı kimlikleri
print("\nCBC katılımcılar:")
print(sorted(cbc_df["SUBJECT_ID"].unique()))

print("\nCMP katılımcılar:")
print(sorted(cmp_df["SUBJECT_ID"].unique()))

# 3. Ölçüm türleri
print("\nCBC ölçüm türleri:")
print(sorted(cbc_df["ANALYTE"].unique()))

print("\nCMP ölçüm türleri:")
print(sorted(cmp_df["ANALYTE"].unique()))

# 4. Zaman noktaları
print("\nCBC zaman noktaları:")
print(sorted(cbc_df["TEST_DATE"].unique()))

print("\nCMP zaman noktaları:")
print(sorted(cmp_df["timepoint"].unique()))

# 5. Her ölçümde kaç kayıt var?
print("\nCBC ölçüm kayıt sayıları:")
display(cbc_df.groupby("ANALYTE").size().sort_values(ascending=False))

print("\nCMP ölçüm kayıt sayıları:")
display(cmp_df.groupby("ANALYTE").size().sort_values(ascending=False))

# 6. CMP eksik VALUE değerlerini ölçüme göre incele
print("\nCMP eksik VALUE dağılımı:")
display(
    cmp_df[cmp_df["VALUE"].isna()]
    .groupby("ANALYTE")
    .size()
    .sort_values(ascending=False)
)

# 7. Tekrarlanan kayıt kontrolü
print("\nCBC tekrarlanan kayıt sayısı:",
      cbc_df.duplicated().sum())

print("CMP tekrarlanan kayıt sayısı:",
      cmp_df.duplicated().sum())



In [ ]:

# 1. Katılımcıların cinsiyet bilgileri
print("CBC katılımcı ve cinsiyet bilgileri:")
display(
    cbc_df[["SUBJECT_ID", "SEX"]]
    .drop_duplicates()
    .sort_values("SUBJECT_ID")
)

# 2. FEMALE ölçümlerinin hangi kayıtlara ait olduğunu incele
print("\nCBC FEMALE ölçümleri:")
display(
    cbc_df[
        cbc_df["ANALYTE"].str.contains("FEMALE", na=False)
    ][["ANALYTE", "VALUE", "UNITS", "SUBJECT_ID", "SEX", "TEST_DATE"]]
)

# 3. Her ölçümün birimini incele
print("\nCBC ölçüm ve birimleri:")
display(
    cbc_df[["ANALYTE", "UNITS"]]
    .drop_duplicates()
    .sort_values("ANALYTE")
)

print("\nCMP ölçüm ve birimleri:")
display(
    cmp_df[["ANALYTE", "UNITS"]]
    .drop_duplicates()
    .sort_values("ANALYTE")
)

# 4. Her katılımcının her ölçüm için geçerli kayıt sayısı
print("\nCBC katılımcı bazında ölçüm sayıları:")
display(
    cbc_df.groupby(["SUBJECT_ID", "ANALYTE"])["VALUE"]
    .count()
    .unstack(fill_value=0)
)

print("\nCMP katılımcı bazında ölçüm sayıları:")
display(
    cmp_df.groupby(["SUBJECT_ID", "ANALYTE"])["VALUE"]
    .count()
    .unstack(fill_value=0)
)

# 5. Eksik CMP kayıtlarının ayrıntısı
print("\nCMP eksik VALUE kayıtları:")
display(
    cmp_df[cmp_df["VALUE"].isna()]
)


In [ ]:

import matplotlib.pyplot as plt

# Görev zamanlarının doğru sıralanması
time_order = ["L-92", "L-44", "L-3", "R+1", "R+45", "R+82", "R+194"]

# Grafik oluşturacağımız ölçümler
measurements = [
    {
        "data": cbc_df,
        "analyte": "WHITE BLOOD CELL COUNT",
        "time_col": "TEST_DATE",
        "title": "White Blood Cell Count",
    },
    {
        "data": cbc_df,
        "analyte": "PLATELET COUNT",
        "time_col": "TEST_DATE",
        "title": "Platelet Count",
    },
    {
        "data": cmp_df,
        "analyte": "GLUCOSE",
        "time_col": "timepoint",
        "title": "Blood Glucose",
    }
]

for item in measurements:

    df = item["data"]

    selected = df[df["ANALYTE"] == item["analyte"]].copy()

    selected[item["time_col"]] = pd.Categorical(
        selected[item["time_col"]],
        categories=time_order,
        ordered=True
    )

    plt.figure(figsize=(10, 5))

    for subject in sorted(selected["SUBJECT_ID"].unique()):

        subject_data = selected[
            selected["SUBJECT_ID"] == subject
        ].sort_values(item["time_col"])

        plt.plot(
            subject_data[item["time_col"]].astype(str),
            subject_data["VALUE"],
            marker="o",
            label=subject
        )

    plt.title(item["title"])
    plt.xlabel("Mission Timepoint")
    plt.ylabel(selected["UNITS"].iloc[0])
    plt.legend(title="Participant")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()



In [ ]:

# 1. C003 katılımcısının dikkat çeken trombosit ölçümünü bul
print("C003 - Platelet Count kayıtları:")

display(
    cbc_df[
        (cbc_df["SUBJECT_ID"] == "C003") &
        (cbc_df["ANALYTE"] == "PLATELET COUNT")
    ][
        ["SUBJECT_ID", "ANALYTE", "VALUE",
         "RANGE_MIN", "RANGE_MAX", "UNITS", "TEST_DATE"]
    ].sort_values("TEST_DATE")
)

# 2. R+45 zamanındaki glukoz değerlerini incele
print("\nR+45 - Glucose kayıtları:")

display(
    cmp_df[
        (cmp_df["ANALYTE"] == "GLUCOSE") &
        (cmp_df["timepoint"] == "R+45")
    ][
        ["SUBJECT_ID", "ANALYTE", "VALUE",
         "RANGE_MIN", "RANGE_MAX", "UNITS", "timepoint"]
    ]
)

# 3. Seçilen ölçümlerde referans aralığı dışındaki değerleri bul
def check_reference_range(df, analyte, time_column):

    selected = df[df["ANALYTE"] == analyte].copy()

    outside = selected[
        (selected["VALUE"] < selected["RANGE_MIN"]) |
        (selected["VALUE"] > selected["RANGE_MAX"])
    ]

    print(f"\n{analyte} - Referans aralığı dışında görünen kayıtlar:")
    
    display(
        outside[
            ["SUBJECT_ID", "ANALYTE", "VALUE",
             "RANGE_MIN", "RANGE_MAX", "UNITS", time_column]
        ]
    )

check_reference_range(cbc_df, "PLATELET COUNT", "TEST_DATE")
check_reference_range(cmp_df, "GLUCOSE", "timepoint")


In [ ]:

# TRANSFORMED dosyalarını oku

CBC_TRANSFORMED_FILE = (
    PROJECT_ROOT / "data" / "raw" / "OSD-569" /
    "LSDS-7_Complete_Blood_Count_CBC_TRANSFORMED.csv"
)

CMP_TRANSFORMED_FILE = (
    PROJECT_ROOT / "data" / "raw" / "OSD-575" /
    "LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANSFORMED.csv"
)

cbc_transformed = pd.read_csv(CBC_TRANSFORMED_FILE)
cmp_transformed = pd.read_csv(CMP_TRANSFORMED_FILE)

# Trombosit değerini kontrol et
print("CBC TRANSFORMED - C003 Platelet Count")

display(
    cbc_transformed[
        (cbc_transformed["SUBJECT_ID"] == "C003") &
        (cbc_transformed["ANALYTE"] == "PLATELET COUNT")
    ]
)

# Glukoz değerlerini kontrol et
print("\nCMP TRANSFORMED - R+45 Glucose")

display(
    cmp_transformed[
        (cmp_transformed["ANALYTE"] == "GLUCOSE") &
        (cmp_transformed["timepoint"] == "R+45")
    ]
)


In [ ]:

# TRANSFORMED dosyalarını oku

CBC_TRANSFORMED_FILE = (
    PROJECT_ROOT / "data" / "raw" / "OSD-569" /
    "LSDS-7_Complete_Blood_Count_CBC_TRANSFORMED.csv"
)

CMP_TRANSFORMED_FILE = (
    PROJECT_ROOT / "data" / "raw" / "OSD-575" /
    "LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANSFORMED.csv"
)

cbc_transformed = pd.read_csv(CBC_TRANSFORMED_FILE)
cmp_transformed = pd.read_csv(CMP_TRANSFORMED_FILE)

# Trombosit değerini kontrol et
print("CBC TRANSFORMED - C003 Platelet Count")

display(
    cbc_transformed[
        (cbc_transformed["SUBJECT_ID"] == "C003") &
        (cbc_transformed["ANALYTE"] == "PLATELET COUNT")
    ]
)

# Glukoz değerlerini kontrol et
print("\nCMP TRANSFORMED - R+45 Glucose")

display(
    cmp_transformed[
        (cmp_transformed["ANALYTE"] == "GLUCOSE") &
        (cmp_transformed["timepoint"] == "R+45")
    ]
)


In [ ]:
# TRANSFORMED dosyalarını oku

CBC_TRANSFORMED_FILE = (
    PROJECT_ROOT / "data" / "raw" / "OSD-569" /
    "LSDS-7_Complete_Blood_Count_CBC_TRANSFORMED.csv"
)

CMP_TRANSFORMED_FILE = (
    PROJECT_ROOT / "data" / "raw" / "OSD-575" /
    "LSDS-8_Comprehensive_Metabolic_Panel_CMP_TRANSFORMED.csv"
)

cbc_transformed = pd.read_csv(CBC_TRANSFORMED_FILE)
cmp_transformed = pd.read_csv(CMP_TRANSFORMED_FILE)

print("CBC TRANSFORMED yüklendi:", cbc_transformed.shape)
print("CMP TRANSFORMED yüklendi:", cmp_transformed.shape)

In [ ]:

# CBC: C003 trombosit değerini karşılaştır
print("CBC - SUBMITTED")
display(
    cbc_df[
        (cbc_df["SUBJECT_ID"] == "C003") &
        (cbc_df["ANALYTE"] == "PLATELET COUNT") &
        (cbc_df["TEST_DATE"] == "L-92")
    ]
)

print("CBC - TRANSFORMED")
display(
    cbc_transformed[
        (cbc_transformed["SUBJECT_ID"] == "C003") &
        (cbc_transformed["ANALYTE"] == "PLATELET COUNT")
    ]
)

# CMP: R+45 glukoz değerlerini karşılaştır
print("CMP - SUBMITTED")
display(
    cmp_df[
        (cmp_df["ANALYTE"] == "GLUCOSE") &
        (cmp_df["timepoint"] == "R+45")
    ]
)

print("CMP - TRANSFORMED")
display(
    cmp_transformed[
        (cmp_transformed["ANALYTE"] == "GLUCOSE")
    ]
)


In [ ]:

print("CBC TRANSFORMED sütunları:")
print(cbc_transformed.columns.tolist())

print("\nCBC TRANSFORMED ilk 5 satır:")
display(cbc_transformed.head())

print("\nCMP TRANSFORMED sütunları:")
print(cmp_transformed.columns.tolist())

print("\nCMP TRANSFORMED ilk 5 satır:")
display(cmp_transformed.head())



In [ ]:

# 1. CBC: C003 - L-92 trombosit değerini bul

print("CBC TRANSFORMED - C003 L-92")

display(
    cbc_transformed[
        cbc_transformed["Sample Name"].str.contains(
            "C003_whole-blood_L-92",
            case=False,
            na=False
        )
    ][[
        "Sample Name",
        "platelet_count_value_thousand_per_microliter",
        "platelet_count_range_min_thousand_per_microliter",
        "platelet_count_range_max_thousand_per_microliter"
    ]]
)


# 2. CMP: R+45 glukoz değerlerini bul

print("\nCMP TRANSFORMED - R+45 GLUCOSE")

display(
    cmp_transformed[
        cmp_transformed["Sample Name"].str.contains(
            "R+45",
            regex=False,
            na=False
        )
    ][[
        "Sample Name",
        "glucose_value_milligram_per_deciliter",
        "glucose_range_min_milligram_per_deciliter",
        "glucose_range_max_milligram_per_deciliter"
    ]]
)



In [ ]:

import zipfile

metadata_files = [
    PROJECT_ROOT / "data" / "raw" / "OSD-569" /
    "OSD-569_metadata_OSD-569-ISA.zip",

    PROJECT_ROOT / "data" / "raw" / "OSD-575" /
    "OSD-575_metadata_OSD-575-ISA.zip"
]

for zip_path in metadata_files:

    print("\n" + "=" * 60)
    print(zip_path.name)
    print("=" * 60)

    with zipfile.ZipFile(zip_path, "r") as archive:

        for file_name in archive.namelist():
            print(file_name)


In [ ]:

import zipfile

metadata_targets = {
    "OSD-569": "a_OSD-569_complete-blood-count_hematology-analyzer_Quest Diagnostics.txt",
    "OSD-575": "a_OSD-575_analytical-chemistry_comprehensive-metabolic-panel_quest diagnostics.txt"
}

for study, target_file in metadata_targets.items():

    zip_path = (
        PROJECT_ROOT / "data" / "raw" / study /
        f"{study}_metadata_{study}-ISA.zip"
    )

    print("\n" + "=" * 70)
    print(study, "-", target_file)
    print("=" * 70)

    with zipfile.ZipFile(zip_path, "r") as archive:
        content = archive.read(target_file).decode(
            "utf-8",
            errors="replace"
        )

        print(content)


In [ ]:

import zipfile

for study in ["OSD-569", "OSD-575"]:

    zip_path = (
        PROJECT_ROOT / "data" / "raw" / study /
        f"{study}_metadata_{study}-ISA.zip"
    )

    print("\n" + "=" * 70)
    print(study, "- STUDY DESIGN")
    print("=" * 70)

    with zipfile.ZipFile(zip_path, "r") as archive:

        for file_name in [
            f"s_{study}.txt",
            "i_Investigation.txt"
        ]:
            print(f"\n--- {file_name} ---")

            content = archive.read(file_name).decode(
                "utf-8",
                errors="replace"
            )

            print(content)


In [ ]:

import zipfile
import pandas as pd
from io import BytesIO

for study in ["OSD-569", "OSD-575"]:

    zip_path = (
        PROJECT_ROOT / "data" / "raw" / study /
        f"{study}_metadata_{study}-ISA.zip"
    )

    with zipfile.ZipFile(zip_path, "r") as archive:

        file_name = f"s_{study}.txt"

        df = pd.read_csv(
            BytesIO(archive.read(file_name)),
            sep="\t"
        )

    print("\n", "=" * 60)
    print(study)
    print("=" * 60)

    columns_to_show = [
        "Source Name",
        "Sample Name",
        "Characteristics[Sex]",
        "Characteristics[Material Type]",
        "Factor Value[Time]",
        "Factor Value[Spaceflight]",
        "Parameter Value[Sample Preservation Method]",
        "Parameter Value[Sample Storage Temperature]"
    ]

    available_columns = [
        col for col in columns_to_show
        if col in df.columns
    ]

    display(df[available_columns].head(10))

    print("\nKatılımcı sayısı:", df["Source Name"].nunique())
    print("Örnek sayısı:", df["Sample Name"].nunique())
    print("Zaman noktaları:")
    print(df["Factor Value[Time]"].value_counts())


In [ ]:

# CBC metadata örneklerini katılımcı ve zaman noktasına göre incele

import zipfile
import pandas as pd
from io import BytesIO

zip_path = (
    PROJECT_ROOT / "data" / "raw" / "OSD-569" /
    "OSD-569_metadata_OSD-569-ISA.zip"
)

with zipfile.ZipFile(zip_path, "r") as archive:
    cbc_metadata = pd.read_csv(
        BytesIO(archive.read("s_OSD-569.txt")),
        sep="\t"
    )

# Katılımcı ve zaman noktası bazında örnek sayısı
sample_counts = (
    cbc_metadata
    .groupby(["Source Name", "Factor Value[Time]"])
    ["Sample Name"]
    .count()
    .unstack(fill_value=0)
)

display(sample_counts)

# Aynı katılımcı ve zamanda birden fazla örnek var mı?
duplicates = cbc_metadata[
    cbc_metadata.duplicated(
        subset=["Source Name", "Factor Value[Time]"],
        keep=False
    )
].sort_values(["Source Name", "Factor Value[Time]"])

print("Birden fazla örneği bulunan kayıtlar:")
display(
    duplicates[
        ["Source Name", "Sample Name", "Factor Value[Time]"]
    ]
)


In [ ]:

# TRANSFORMED dosyasındaki örnek adları
cbc_sample_names = set(cbc_transformed["Sample Name"].dropna())

# Metadata içinden yalnızca bu örnekleri seç
cbc_matched_metadata = cbc_metadata[
    cbc_metadata["Sample Name"].isin(cbc_sample_names)
].copy()

print("TRANSFORMED örnek sayısı:", len(cbc_sample_names))
print("Metadata ile eşleşen örnek sayısı:", len(cbc_matched_metadata))

print("\nEşleşmeyen TRANSFORMED örnekleri:")
print(
    sorted(
        cbc_sample_names -
        set(cbc_matched_metadata["Sample Name"])
    )
)

print("\nEşleşen metadata:")
display(
    cbc_matched_metadata[
        [
            "Source Name",
            "Sample Name",
            "Factor Value[Time]",
            "Factor Value[Spaceflight]"
        ]
    ].sort_values(["Source Name", "Factor Value[Time]"])
)


In [ ]:

# CMP TRANSFORMED örneklerini metadata ile eşleştir

cmp_sample_names = set(cmp_transformed["Sample Name"].dropna())

cmp_matched_metadata = cmp_metadata[
    cmp_metadata["Sample Name"].isin(cmp_sample_names)
].copy()

print("TRANSFORMED örnek sayısı:", len(cmp_sample_names))
print("Metadata ile eşleşen örnek sayısı:", len(cmp_matched_metadata))

print("\nEşleşmeyen TRANSFORMED örnekleri:")
print(
    sorted(
        cmp_sample_names -
        set(cmp_matched_metadata["Sample Name"])
    )
)

print("\nEşleşen metadata:")
display(
    cmp_matched_metadata[
        [
            "Source Name",
            "Sample Name",
            "Factor Value[Time]",
            "Factor Value[Spaceflight]"
        ]
    ].sort_values(["Source Name", "Factor Value[Time]"])
)


In [ ]:

import zipfile
import pandas as pd
from io import BytesIO

cmp_zip_path = (
    PROJECT_ROOT / "data" / "raw" / "OSD-575" /
    "OSD-575_metadata_OSD-575-ISA.zip"
)

with zipfile.ZipFile(cmp_zip_path, "r") as archive:
    cmp_metadata = pd.read_csv(
        BytesIO(archive.read("s_OSD-575.txt")),
        sep="\t"
    )

print("CMP metadata yüklendi.")
print("Örnek sayısı:", len(cmp_metadata))


In [ ]:

cmp_sample_names = set(cmp_transformed["Sample Name"].dropna())

cmp_matched_metadata = cmp_metadata[
    cmp_metadata["Sample Name"].isin(cmp_sample_names)
].copy()

print("TRANSFORMED örnek sayısı:", len(cmp_sample_names))
print("Metadata ile eşleşen örnek sayısı:", len(cmp_matched_metadata))

print("\nEşleşmeyen örnekler:")
print(
    sorted(
        cmp_sample_names -
        set(cmp_matched_metadata["Sample Name"])
    )
)

display(
    cmp_matched_metadata[
        [
            "Source Name",
            "Sample Name",
            "Factor Value[Time]",
            "Factor Value[Spaceflight]"
        ]
    ].sort_values(["Source Name", "Factor Value[Time]"])
)
